# Payroll Change Exploratory Analysis

## Can the change in payroll costs since the start of last year be explained by the increase in headcount over the same period? Are there any other contributing factors?

## 1. Import SQL results from BigQuery

In [1]:
import pandas as pd
from google.cloud import bigquery

client = bigquery.Client(
    project="employee-data-project-507314"
)

#This query brings in a joined table acorss payroll entries, payroll runs, employees, and department tables

query = """
SELECT 
    pe.payroll_entry_id,
    pe.payroll_run_id,
    pe.employee_id,
    pe.gross_pay,
    pe.overtime_pay,
    pe.bonus_pay,
    pr.pay_period,
    d.department_name
FROM 
    `employee-data-project-507314.employee_dataset.payroll_entries` pe

JOIN
    `employee-data-project-507314.employee_dataset.payroll_runs` pr
ON
    pe.payroll_run_id = pr.payroll_run_id

JOIN
    `employee-data-project-507314.employee_dataset.employees` e
ON
    pe.employee_id = e.employee_id

JOIN
    `employee-data-project-507314.employee_dataset.departments` d
ON
    e.department_id = d.department_id
"""

# Convert query results to pd dataframe

df = client.query(query).to_dataframe()

### 1.1. Display head of table to validate correct pull-through

In [2]:
df.head()

,payroll_entry_id,payroll_run_id,employee_id,gross_pay,overtime_pay,bonus_pay,pay_period,department_name
0,1,1,1,3586.06,252.73,0.00,2025-01,Operations
1,2,2,1,3632.34,299.01,0.00,2025-02,Operations
2,3,3,1,3409.01,61.79,13.89,2025-03,Operations
3,4,4,1,3492.22,158.89,0.00,2025-04,Operations
4,5,5,1,3510.80,177.47,0.00,2025-05,Operations


## 2. Exploratory Data Analysis - To what extent does departmental gross payroll growth track growth in the number of employees paid, and which departments warrant further investigation?

### 2.1. Build a table which pulls through total pay from first and last payroll runs, with a % change column.

In [3]:
# Identify the first and latest payroll run. 
# This ensures that if a new payroll run is added, this will always take the last one.
# This does assume that payroll_run_id index numbers will always be sequential.

first_run = df['payroll_run_id'].min()
latest_run = df['payroll_run_id'].max()

# Create a filtered table to include only first and last payroll run IDs.

filtered_df = df[
    df["payroll_run_id"].isin([first_run, latest_run])
]

# Pull a Pivot Table from this that takes the sum of net pay by department for first and last payrolls.

departmental_payroll = filtered_df.pivot_table(
    index = "department_name",
    columns = "payroll_run_id",
    values = "gross_pay",
    aggfunc = "sum",
    fill_value = 0
)

# Rename columns to be more legible.

departmental_payroll = departmental_payroll.set_axis(["First pay run sum", "Last pay run sum"], axis=1)

# Establish a column which provides percentage change between first and last values.

departmental_payroll['Percentage Change Gross (%)'] = ((
    (departmental_payroll['Last pay run sum']
     - departmental_payroll['First pay run sum'])
    / departmental_payroll['First pay run sum']
) * 100
).round(2)

departmental_payroll.sort_values(by="Percentage Change Gross (%)", ascending=False)

,First pay run sum,Last pay run sum,Percentage Change Gross (%)
department_name,,,
Finance,140221.51,179603.62,28.09
Legal,54377.42,67651.49,24.41
Payroll,185111.47,222142.55,20.00
Customer Support,387637.18,460373.58,18.76
HR,364846.63,419535.24,14.99
Operations,644856.55,738733.82,14.56
IT,176195.28,196166.90,11.33
Marketing,187325.82,204356.78,9.09
Sales,336757.73,336408.07,-0.10


### 2.2. Build a table which does the same, but for headcount.

In [4]:

# Create another Pivot Table to establish employee headcounts at the two payroll runs.

departmental_headcount = filtered_df. pivot_table(
    index = "department_name",
    columns = "payroll_run_id",
    values = "employee_id",
    aggfunc = "nunique",
    fill_value = 0
)

# Rename headcount table columns

departmental_headcount = departmental_headcount.set_axis(["First headcount", "Last headcount"], axis=1)

# Establish headcount percentage change column

departmental_headcount['Percentage Change Headcount (%)'] = ((
    (departmental_headcount['Last headcount']
     - departmental_headcount['First headcount'])
    / departmental_headcount['First headcount']
) * 100
).round(2)

departmental_headcount.sort_values(by="Percentage Change Headcount (%)", ascending=False)

,First headcount,Last headcount,Percentage Change Headcount (%)
department_name,,,
Finance,40,51,27.50
Legal,9,11,22.22
Customer Support,160,191,19.38
Payroll,52,62,19.23
Operations,171,196,14.62
HR,101,112,10.89
IT,46,51,10.87
Marketing,61,67,9.84
Sales,108,108,0.00


### 2.3. Join the tables and add a column which finds the difference in the two percentage columns for each department.

In [5]:
# Join tables together

departmental_payroll = departmental_payroll.join(departmental_headcount)

# Add final column with difference between gross pay growth and headcount growth

departmental_payroll["Payroll Growth minus Headcount Growth"] = (
    departmental_payroll["Percentage Change Gross (%)"] - departmental_payroll["Percentage Change Headcount (%)"]
)

# Display result

departmental_payroll.sort_values(by="Payroll Growth minus Headcount Growth", ascending=False)

,First pay run sum,Last pay run sum,Percentage Change Gross (%),First headcount,Last headcount,Percentage Change Headcount (%),Payroll Growth minus Headcount Growth
department_name,,,,,,,
HR,364846.63,419535.24,14.99,101,112,10.89,4.10
Legal,54377.42,67651.49,24.41,9,11,22.22,2.19
Payroll,185111.47,222142.55,20.00,52,62,19.23,0.77
Finance,140221.51,179603.62,28.09,40,51,27.50,0.59
IT,176195.28,196166.90,11.33,46,51,10.87,0.46
Operations,644856.55,738733.82,14.56,171,196,14.62,-0.06
Sales,336757.73,336408.07,-0.10,108,108,0.00,-0.10
Customer Support,387637.18,460373.58,18.76,160,191,19.38,-0.62
Marketing,187325.82,204356.78,9.09,61,67,9.84,-0.75


### 2.4. Comparison to overall company values

In [6]:
# Calculate company-wide totals from the existing departmental table

first_total_pay = departmental_payroll['First pay run sum'].sum()
latest_total_pay = departmental_payroll['Last pay run sum'].sum()

first_total_headcount = departmental_payroll['First headcount'].sum()
latest_total_headcount = departmental_payroll['Last headcount'].sum()

# Calculate overall percentage changes

global_pay_change = (
    (latest_total_pay - first_total_pay) / first_total_pay
) * 100

global_headcount_change = (
    (latest_total_headcount - first_total_headcount) / first_total_headcount
) * 100

# Compare the two percentage changes

global_pay_vs_headcount = global_pay_change - global_headcount_change

print(f"Global gross payroll change: {global_pay_change:.2f}%")
print(f"Global paid headcount change: {global_headcount_change:.2f}%")
print(f"Payroll growth minus headcount growth: {global_pay_vs_headcount:.2f} percentage points")

Global gross payroll change: 12.19%
Global paid headcount change: 12.16%
Payroll growth minus headcount growth: 0.03 percentage points


## 3. Finding an Average Gross Pay per Employee metric

In [7]:
# Calculate average gross pay per paid employee
departmental_payroll["First Run Avg Gross Pay"] = (
    departmental_payroll["First pay run sum"]
    / departmental_payroll["First headcount"]
).round(2)

departmental_payroll["Last Run Avg Gross Pay"] = (
    departmental_payroll["Last pay run sum"]
    / departmental_payroll["Last headcount"]
).round(2)

departmental_payroll["Avg Gross Pay Change (%)"] = (
    (
    (
    departmental_payroll["Last Run Avg Gross Pay"]
    - departmental_payroll["First Run Avg Gross Pay"]
    )
    / departmental_payroll["First Run Avg Gross Pay"]
    ) * 100
).round(2)

departmental_payroll.sort_values(by = "Avg Gross Pay Change (%)", ascending = False)
    

,First pay run sum,Last pay run sum,Percentage Change Gross (%),First headcount,Last headcount,Percentage Change Headcount (%),Payroll Growth minus Headcount Growth,First Run Avg Gross Pay,Last Run Avg Gross Pay,Avg Gross Pay Change (%)
department_name,,,,,,,,,,
HR,364846.63,419535.24,14.99,101,112,10.89,4.10,3612.34,3745.85,3.70
Legal,54377.42,67651.49,24.41,9,11,22.22,2.19,6041.94,6150.14,1.79
Payroll,185111.47,222142.55,20.00,52,62,19.23,0.77,3559.84,3582.94,0.65
Finance,140221.51,179603.62,28.09,40,51,27.50,0.59,3505.54,3521.64,0.46
IT,176195.28,196166.90,11.33,46,51,10.87,0.46,3830.33,3846.41,0.42
Operations,644856.55,738733.82,14.56,171,196,14.62,-0.06,3771.09,3769.05,-0.05
Sales,336757.73,336408.07,-0.10,108,108,0.00,-0.10,3118.13,3114.89,-0.10
Customer Support,387637.18,460373.58,18.76,160,191,19.38,-0.62,2422.73,2410.33,-0.51
Marketing,187325.82,204356.78,9.09,61,67,9.84,-0.75,3070.92,3050.10,-0.68


## 4. Establishing a similar summary in regard to overtime pay

In [8]:
# Filter to the first and latest payroll runs
overtime_filtered = filtered_df.copy()

# Summarise overtime pay by department and payroll run
overtime_summary = overtime_filtered.pivot_table(
    index='department_name',
    columns='payroll_run_id',
    values='overtime_pay',
    aggfunc='sum',
    fill_value=0
)

# Label the columns clearly
overtime_summary.columns = [
    'First Run Overtime Pay',
    'Last Run Overtime Pay'
]

# Find percentage change over this period

overtime_summary["Change in Overtime Pay (%)"] = (((
    overtime_summary["Last Run Overtime Pay"]
    - overtime_summary["First Run Overtime Pay"]
) / overtime_summary["First Run Overtime Pay"]
) * 100
).round(2)

# Join headcount table

overtime_summary = overtime_summary.join(departmental_headcount)

# Add columns to find average overtime pay for first and last runs

overtime_summary["First Run Avg Overtime Pay"] = (
    overtime_summary["First Run Overtime Pay"]
    / overtime_summary["First headcount"]
).round(2)

overtime_summary["Last Run Avg Overtime Pay"] = (
    overtime_summary["Last Run Overtime Pay"]
    / overtime_summary["Last headcount"]
).round(2)

# Find average average overtime pay change in %

overtime_summary["Avg Overtime Pay Change (%)"] = (((
    overtime_summary["Last Run Avg Overtime Pay"]
    - overtime_summary["First Run Avg Overtime Pay"]
) / overtime_summary["First Run Avg Overtime Pay"]
) * 100
).round(2)

overtime_summary.sort_values(by = "Avg Overtime Pay Change (%)", ascending = False)


,First Run Overtime Pay,Last Run Overtime Pay,Change in Overtime Pay (%),First headcount,Last headcount,Percentage Change Headcount (%),First Run Avg Overtime Pay,Last Run Avg Overtime Pay,Avg Overtime Pay Change (%)
department_name,,,,,,,,,
IT,2861.96,4083.58,42.68,46,51,10.87,62.22,80.07,28.69
Legal,960.75,1484.82,54.55,9,11,22.22,106.75,134.98,26.44
Payroll,3778.13,4975.87,31.70,52,62,19.23,72.66,80.26,10.46
Finance,3138.16,4270.27,36.08,40,51,27.50,78.45,83.73,6.73
HR,8346.62,9785.22,17.24,101,112,10.89,82.64,87.37,5.72
Operations,30106.64,33567.25,11.49,171,196,14.62,176.06,171.26,-2.73
Sales,16424.40,15658.07,-4.67,108,108,0.00,152.08,144.98,-4.67
Customer Support,17970.56,19873.65,10.59,160,191,19.38,112.32,104.05,-7.36
Marketing,4909.14,4606.76,-6.16,61,67,9.84,80.48,68.76,-14.56


## 4.1. Calculate overtime pay as a percentage of total payroll over this time period.

In [9]:
# Create a simple df to show what the share of total payroll is held by overtime

total_gross_pay = df["gross_pay"].sum()
total_overtime = df["overtime_pay"].sum()
total_bonus = df["bonus_pay"].sum()
total_pay = total_gross_pay + total_overtime + total_bonus

overtime_share = pd.DataFrame({
    "Total Payroll": [total_pay],
    "Total Overtime Pay": [total_overtime],
    "Overtime Share (%)": [
        (total_overtime / total_pay) * 100
    ]
})

print(overtime_share.round(2))

   Total Payroll  Total Overtime Pay  Overtime Share (%)
0    51432398.07           1650801.7                3.21


## 4.2. Calculate overtime pay change and total payroll change to show how much of the change can be attributed to overtime.

In [10]:
# Sum each pay component by payroll run

pay_by_run = df.groupby("payroll_run_id")[
    ["gross_pay", "overtime_pay", "bonus_pay"]
    ].sum()

# Calculate total pay for first and latest runs

first_total_pay = pay_by_run.loc[first_run].sum()
latest_total_pay = pay_by_run.loc[latest_run].sum()

# Calculate overtime pay for first and latest runs

first_overtime_pay = pay_by_run["overtime_pay"].loc[first_run].sum()
latest_overtime_pay = pay_by_run["overtime_pay"].loc[latest_run].sum()

# Calculate absolute and percentage changes

total_pay_change = latest_total_pay - first_total_pay
overtime_pay_change = latest_overtime_pay - first_overtime_pay

total_pay_change_pct = (
    total_pay_change / first_total_pay
) * 100

overtime_pay_change_pct = (
    overtime_pay_change / first_overtime_pay
) * 100

# Show changes in a df comparing percentage change in total and overtime pay

pay_change_summary = pd.DataFrame({
    "First run (£)": [first_total_pay, first_overtime_pay],
    "Latest Run (£)": [latest_total_pay, latest_overtime_pay],
    "Change (£)": [total_pay_change, overtime_pay_change],
    "Change (%)": [total_pay_change_pct, overtime_pay_change_pct]
}, index=["Total Pay", "Overtime Pay"])

pay_change_summary.round(2)

,First run (£),Latest Run (£),Change (£),Change (%)
Total Pay,2811266.32,3150804.54,339538.22,12.08
Overtime Pay,93591.54,101860.65,8269.11,8.84


In [15]:
# Calculate overtime's contribution to the change in total pay

overtime_contribution_pct = (
    overtime_pay_change / total_pay_change
    ) * 100

print(f"Change in total pay: £{total_pay_change:,.2f}")
print(f"Change in overtime pay: £{overtime_pay_change:,.2f}")
print(
    "Overtime contribution to total pay change: "
    f"{overtime_contribution_pct:.2f}%"
)

Change in total pay: £339,538.22
Change in overtime pay: £8,269.11
Overtime contribution to total pay change: 2.44%


### Following exploratory analysis, gross pay appears to be consistent with headcount growth, though it is important to understand that the comparison between the first and latest pay runs does not account for fluctuations between those two points. To view a more detailed timeline change comparison, please view the Power BI file in the repo.

### Over the same period, total pay — comprising gross pay, overtime pay and bonus pay — increased by 12.08% (£339,538.22). Overtime pay increased by 8.84% (£8,269.11), accounting for approximately 2.44% of the total increase. This suggests that overtime was not a major contributor to the overall increase in total pay.